# Primeros modelos de mi proyecto (M5, CA_1)

Predecir las unidades vendidas de un item al dia siguiente. Preparacion y division por fecha importadas de `pipeline_proyecto.py` (ultimas 8 semanas de prueba). Entrenamiento con una muestra de 200 000 filas. Metrica: MAE en unidades.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline

from pipeline_proyecto import cargar_y_dividir, construir_preparacion

X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir(maximo_entrenamiento=200_000)
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)

Entrenamiento: (200000, 9) Prueba: (170744, 9)


## Linea base

Una regla del dominio: la demanda de manana es el promedio de los ultimos 28 dias del item (`units_media_28`). Es lo que haria un encargado de tienda sin modelo, y es mas exigente que un `DummyRegressor` (que predice una constante, la mediana, y se muestra tambien como piso).

In [2]:
resultados = {}
entrenados = {}


def registrar(nombre, pred_ent, pred_prueba, segundos):
    mae_ent = mean_absolute_error(y_ent, pred_ent)
    mae_prueba = mean_absolute_error(y_prueba, pred_prueba)
    resultados[nombre] = {"modelo": nombre, "mae_entrenamiento": round(mae_ent, 4),
                          "mae_prueba": round(mae_prueba, 4), "brecha": round(mae_prueba - mae_ent, 4),
                          "segundos": round(segundos, 2)}
    print(f"{nombre:34s} entrenamiento {mae_ent:.4f}   prueba {mae_prueba:.4f}   brecha {mae_prueba - mae_ent:+.4f}   {segundos:6.2f} s")


def evaluar(nombre, modelo):
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", modelo)])
    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    registrar(nombre, pipeline.predict(X_ent), pipeline.predict(X_prueba), segundos)
    entrenados[nombre] = pipeline


evaluar("linea base: mediana constante", DummyRegressor(strategy="median"))
registrar("linea base: media de 28 dias", X_ent["units_media_28"], X_prueba["units_media_28"], 0.0)

linea base: mediana constante      entrenamiento 1.6441   prueba 1.5344   brecha -0.1097     0.15 s
linea base: media de 28 dias       entrenamiento 1.1793   prueba 1.1082   brecha -0.0711     0.00 s


## Modelo lineal

In [3]:
evaluar("ridge alpha=1", Ridge(alpha=1.0))
evaluar("ridge alpha=1000", Ridge(alpha=1000.0))

ridge alpha=1                      entrenamiento 1.1921   prueba 1.1190   brecha -0.0731     0.18 s


ridge alpha=1000                   entrenamiento 1.1915   prueba 1.1189   brecha -0.0726     0.18 s


## Ensamblajes

In [4]:
evaluar("bosque sin limite", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
evaluar("bosque min_samples_leaf=50", RandomForestRegressor(n_estimators=100, min_samples_leaf=50, random_state=42, n_jobs=-1))
evaluar("gradient boosting", GradientBoostingRegressor(random_state=42))

bosque sin limite                  entrenamiento 0.4529   prueba 1.1630   brecha +0.7100    10.04 s


bosque min_samples_leaf=50         entrenamiento 1.1233   prueba 1.0907   brecha -0.0326     6.45 s


gradient boosting                  entrenamiento 1.1470   prueba 1.0895   brecha -0.0575    15.23 s


## Tabla de resultados

In [5]:
tabla = pd.DataFrame(resultados.values())
tabla.to_csv("resultados_proyecto.csv", index=False)
tabla

,modelo,mae_entrenamiento,mae_prueba,brecha,segundos
0,linea base: mediana constante,1.6441,1.5344,-0.1097,0.15
1,linea base: media de 28 dias,1.1793,1.1082,-0.0711,0.00
2,ridge alpha=1,1.1921,1.1190,-0.0731,0.18
3,ridge alpha=1000,1.1915,1.1189,-0.0726,0.18
4,bosque sin limite,0.4529,1.1630,0.7100,10.04
5,bosque min_samples_leaf=50,1.1233,1.0907,-0.0326,6.45
6,gradient boosting,1.1470,1.0895,-0.0575,15.23
